# Exploratory Data Analysis Before Cleaning

Inspect the full `merged_order_items` dataset from `data/interim` before deciding how to clean it.
Each row represents one order item, identified by `(order_id, order_item_id)`; order attributes can repeat across items.

- Use only the full merged dataset through the existing schema-aware loader. Do not reload raw tables or rerun merging.
- Keep all statuses, missing values and unusual records for inspection. Do not filter, fill, normalize or overwrite the input.
- Calculate diagnostic summaries in separate objects. Use the appropriate item, order or entity denominator for each question.
- Display tables and plots in this notebook. Do not export datasets or generate/update documentation.
- Business feature engineering, preprocessing and delivery-performance analysis belong to later work.


## Project setup

Run the notebook from the repository root or a directory inside the repository.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


# Reuse the project-root discovery convention from the merge notebook.
def find_project_root(start_path: Path) -> Path:
    """Find the project root containing src/ and requirements.txt."""
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path

    raise FileNotFoundError(
        "Could not locate project root containing both 'src/' and 'requirements.txt'."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from src.data_loader import load_data
from src.eda.checks import (
    build_entity_views,
    profile_entity_consistency,
    profile_keys,
)
from src.eda.profiling import get_column_roles, profile_columns, profile_dataset
from src.paths import INTERIM_DIR
from src.pipeline.contracts.schema import BENCHMARK_ROW_COUNT, FULL_MERGE_SCHEMA

print(f"Project root: {PROJECT_ROOT}")
print(f"Interim data path: {INTERIM_DIR}")


## Load the full input

The default reader tries Parquet first, then falls back to CSV if needed, restoring the registered schema.
To select CSV directly, add `file_format="csv"` to the call below.

Keep `input_baseline` in memory for the final non-mutation check. Its row hashes include the index; columns and dtypes are recorded separately.
No baseline file or run metadata is written.


In [ ]:
df = load_data("merged_order_items", stage="interim")

# Record the loaded input without making a second copy of the full dataset.
input_baseline = {
    "shape": df.shape,
    "columns": df.columns.copy(),
    "dtypes": df.dtypes.copy(),
    "row_hashes": pd.util.hash_pandas_object(df, index=True).copy(),
}


## Input baseline

Reference counts describe the current Olist full merge, rather than a cleaning rule.
Show observed counts beside them; investigate differences without modifying the input to match.
The loader already validates the registered schema. A unique-order count does not change the item-level grain of `df`.


In [ ]:
baseline_summary = pd.DataFrame(
    [
        ["Item rows", BENCHMARK_ROW_COUNT, len(df)],
        ["Columns", len(FULL_MERGE_SCHEMA.columns), len(df.columns)],
        ["Unique orders", 98_666, df["order_id"].nunique()],
    ],
    columns=["Metric", "Reference", "Observed"],
)
baseline_summary["Matches reference"] = baseline_summary["Reference"].eq(
    baseline_summary["Observed"]
)
display(baseline_summary)

if not baseline_summary["Matches reference"].all():
    print("Reference counts differ. Check the input version before interpreting later profiles.")

print("Input baseline retained in memory for the final non-mutation check.")


## Initial inspection

Preview the loaded table before examining its structural profiles below.


In [ ]:
df.head()

## B. Structural profiles

All completeness counts on `df` use item rows. Repeated order, customer, product and seller attributes therefore receive item-level weighting here.
Entity views at the end of this section provide one row per observed entity for subsequent distributions.
`customer_id` identifies the customer record attached to a purchase; the full merge does not contain `customer_unique_id`, so customer counts are not unique-person counts.

### Dataset overview

Exact duplicate rows are different from repeated IDs. Participants include every row in a duplicate group; excess copies exclude its first row.


In [ ]:
dataset_profile = profile_dataset(df)
display(dataset_profile)


### Column roles and completeness

Roles reuse the registered merge schema. `source_table` is the first source contributing a column; `source_key` describes that source's grain, not a new key for the full table.
Identifiers and ZIP codes are not numeric measurements.

Missing means `isna()`. Blank means an empty or whitespace-only string and is counted separately.
Percentages use all input item rows. Distinct non-null counts include blanks and preserve original case/whitespace.


In [ ]:
column_roles = get_column_roles()
column_profile = profile_columns(df)

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("Registered column roles")
    display(column_roles)
    print("Completeness at item grain")
    display(column_profile)


### Key structure

Only `(order_id, order_item_id)` must be unique in the full item table. Repeated order/customer/product/seller IDs are expected at this grain.
Missing and blank key-row counts can overlap for composite keys; `invalid_rows` counts their union.
Distinct/repeated-key statistics use valid rows only. `excess_key_rows` is the number of valid rows beyond the first occurrence of each key.


In [ ]:
key_profile = profile_keys(df)
with pd.option_context("display.max_columns", None):
    display(key_profile)


### Entity attribute consistency

For each valid order/customer/product/seller ID, check every attribute contributed by its source table.
A missing value paired with a non-missing value is a conflict; an attribute missing on every row of the same ID is consistent but incomplete.
Null/blank ID rows are reported separately. Conflict percentages use valid distinct entities, rather than item rows.


In [ ]:
entity_consistency = profile_entity_consistency(df)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("Consistency by entity")
    display(entity_consistency["summary"])
    print("Consistency by attribute")
    display(entity_consistency["attributes"])

if entity_consistency["conflicts"].empty:
    print("No conflicting entity attributes found.")
else:
    print("Conflicting entity attributes (all detected entity/attribute pairs)")
    display(entity_consistency["conflicts"])


### Validated entity views in memory

Build source-grain views only after checking complete keys and consistent entity attributes.
If a conflict or invalid key exists, view construction raises an error instead of silently selecting a value.

The views contain entities observed in the full merge, not all records from the raw source tables.
Removing repeated keys inside these temporary, validated views does not clean or modify `df`. Nothing is saved to disk.


In [ ]:
entity_views = build_entity_views(df)
entity_view_summary = pd.DataFrame(
    [
        [name, len(view), len(view.columns)]
        for name, view in entity_views.items()
    ],
    columns=["View", "Observed rows", "Columns"],
)
display(entity_view_summary)


### Input preservation check

Compare the table against the baseline recorded immediately after loading, including column order, dtypes, values and row index.
The same baseline remains available for later EDA sections.


In [ ]:
assert df.shape == input_baseline["shape"], "Input shape changed during profiling."
pd.testing.assert_index_equal(df.columns, input_baseline["columns"])
pd.testing.assert_series_equal(df.dtypes, input_baseline["dtypes"])
pd.testing.assert_series_equal(
    pd.util.hash_pandas_object(df, index=True), input_baseline["row_hashes"],
)
print("Input preserved after structural profiling.")